# LlamaIndex와 AgentCore 메모리 - 투자 포트폴리오 어드바이저 (단기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 투자 포트폴리오 어드바이저를 만드는 방법을 보여줍니다. 단일 고객 상담 세션 내에서의 **단기 메모리** 지속성에 초점을 맞추며, 금융 자문 세션 전반에 걸쳐 고객 프로필, 포트폴리오 보유 현황, 시장 분석 및 투자 추천을 기억할 수 있도록 합니다.

## 아키텍처 개요

![LlamaIndex AgentCore 단기 메모리 아키텍처](LlamaIndex-AgentCore-STM-Arch.png)

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형 메모리                                                               |
| 에이전트 사용 사례  | 투자 포트폴리오 어드바이저                                                       |
| 에이전트 프레임워크 | LlamaIndex                                                                       |
| LLM 모델            | Anthropic Claude 3.7 Sonnet                                                       |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, LlamaIndex 에이전트, 금융 분석 도구                       |
| 예제 난이도         | 중급                                                                             |

학습 내용:
- 금융 자문 데이터를 위한 AgentCore 메모리 생성
- 투자 워크플로를 위한 LlamaIndex 네이티브 메모리 통합 사용
- 포트폴리오 분석을 위한 금융 전용 도구 구축
- 단일 자문 세션 내에서 금융 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 컨텍스트

이 예제에서는 금융 어드바이저가 단일 자문 세션 내에서 고객 포트폴리오를 분석하고, 위험 지표를 평가하며, 투자 추천을 제공하는 "투자 포트폴리오 어드바이저"를 만듭니다. 이 어드바이저는 AgentCore 메모리를 사용하여 상담 전반에 걸쳐 고객 프로필, 포트폴리오 보유 현황, 시장 조사 및 투자 분석에 대한 컨텍스트를 유지합니다.

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 의존성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore 메모리 구성

투자 어드바이저를 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# AgentCore 메모리 리소스 생성
region = os.getenv('AWS_REGION', 'us-east-1')
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f'InvestmentAdvisorShortTerm_{int(datetime.now().timestamp())}',
        description='Investment portfolio advisor short-term memory for single session context',
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )
    memory_id = response['id']
    print(f"AgentCore 메모리 생성 완료: {memory_id}")
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 금융 분석 도구 구현

투자 자문 작업을 위한 전문 도구를 정의합니다:

In [ ]:
def profile_client_risk(client_name: str, risk_tolerance: str, time_horizon: str, investment_goals: str) -> str:
    """Profile client risk tolerance and investment objectives"""
    print(f"고객 프로필: {client_name} (위험 허용도: {risk_tolerance}, 투자 기간: {time_horizon})")
    return f"Profiled client: {client_name}"

def analyze_portfolio_holdings(portfolio_value: str, asset_allocation: str, top_holdings: str) -> str:
    """Analyze current portfolio holdings and allocation"""
    print(f"포트폴리오 분석: ${portfolio_value} 총 가치, 배분: {asset_allocation}")
    return f"Analyzed portfolio worth ${portfolio_value}"

def calculate_risk_metrics(var_95: str, sharpe_ratio: str, beta: str, volatility: str) -> str:
    """Calculate portfolio risk metrics and performance indicators"""
    print(f"위험 지표: VaR 95% {var_95}, 샤프 비율 {sharpe_ratio}, 베타 {beta}, 변동성 {volatility}")
    return f"Calculated risk metrics for portfolio"

def research_market_sector(sector: str, outlook: str, key_drivers: str, recommendation: str) -> str:
    """Research market sector with outlook and investment recommendation"""
    print(f"섹터 리서치: {sector} - {outlook} 전망 ({recommendation})")
    return f"Researched {sector} sector"

def generate_investment_recommendation(security: str, action: str, rationale: str, target_allocation: str) -> str:
    """Generate investment recommendation with rationale"""
    print(f"투자 추천: {action} {security} (목표 배분: {target_allocation})")
    return f"Generated recommendation: {action} {security}"

def check_regulatory_compliance(rule_type: str, compliance_status: str, notes: str) -> str:
    """Check regulatory compliance for investment recommendations"""
    print(f"규정 준수 확인: {rule_type} - {compliance_status}")
    return f"Checked compliance: {rule_type}"

# 에이전트용 도구 객체 생성
financial_tools = [
    FunctionTool.from_defaults(fn=profile_client_risk),
    FunctionTool.from_defaults(fn=analyze_portfolio_holdings),
    FunctionTool.from_defaults(fn=calculate_risk_metrics),
    FunctionTool.from_defaults(fn=research_market_sector),
    FunctionTool.from_defaults(fn=generate_investment_recommendation),
    FunctionTool.from_defaults(fn=check_regulatory_compliance)
]

## 4단계: LlamaIndex 에이전트 구현

단기 메모리 컨텍스트를 가진 투자 어드바이저 에이전트를 생성합니다:

In [ ]:
# 단기 메모리 구성 (단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="financial-advisor",
    memory_id=memory_id,
    session_id="advisory-session-today",  # 전체에서 동일한 세션
    namespace="/investment-advisory/"
)

# AgentCore 메모리 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 투자 어드바이저 에이전트 생성
investment_agent = FunctionAgent(
    tools=financial_tools,
    llm=llm,
    verbose=True
)

print("단기 메모리가 적용된 투자 포트폴리오 어드바이저 준비 완료!")

## 5단계: 단기 메모리 기능 테스트

포괄적인 고객 자문 세션을 통해 투자 어드바이저의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 고객 온보딩 및 위험 프로파일링

In [ ]:
# 고객 세부 정보로 자문 세션 초기화
# (금융 어드바이저 Michael Chen이 고객 Robert Johnson의 위험 프로필 작성 - 중간 위험 허용도, 15년 투자 기간)
response = await investment_agent.run(
    "I'm Financial Advisor Michael Chen meeting with client 'Robert Johnson'. "
    "Profile client risk: 'Robert Johnson' with 'moderate' risk tolerance, '15 years' time horizon, "
    "and investment goals 'retirement planning, wealth preservation, moderate growth'.",
    memory=agentcore_memory
)

print("고객 온보딩:")
print(response)

### 테스트 2: 포트폴리오 보유 현황 분석

In [ ]:
# 현재 포트폴리오 구성 분석
# (포트폴리오 가치 $2,500,000, 자산 배분 주식 60%/채권 35%/현금 5%, 상위 보유 종목 분석)
response = await investment_agent.run(
    "Analyze portfolio holdings: portfolio value '$2,500,000', asset allocation '60% stocks, 35% bonds, 5% cash', "
    "top holdings 'AAPL 8%, MSFT 7%, SPY 12%, BND 15%, VTIAX 10%'.",
    memory=agentcore_memory
)

print("포트폴리오 분석:")
print(response)

### 테스트 3: 위험 지표 계산

In [ ]:
# 종합 위험 지표 계산
# (VaR 95% -$125,000, 샤프 비율 1.15, 베타 0.85, 변동성 12.3% - 중간 위험 프로필로 양호한 위험 조정 수익)
response = await investment_agent.run(
    "Calculate risk metrics: VaR 95% '-$125,000', Sharpe ratio '1.15', Beta '0.85', volatility '12.3%'. "
    "Portfolio shows moderate risk profile with good risk-adjusted returns.",
    memory=agentcore_memory
)

print("위험 지표:")
print(response)

### 테스트 4: 고객 프로필 회상

In [ ]:
# 고객 정보 및 포트폴리오 회상 테스트
# (어떤 고객을 상담 중인지, 위험 허용도, 투자 목표, 현재 포트폴리오 가치 질문)
response = await investment_agent.run(
    "What client am I advising? What are their risk tolerance, investment goals, and current portfolio value?",
    memory=agentcore_memory
)

print("고객 프로필 회상:")
print(response)
print("\n예상 결과: Robert Johnson, 중간 위험, 15년 기간, $2.5M 포트폴리오, 은퇴 계획")

### 테스트 5: 시장 섹터 리서치

In [ ]:
# 잠재적 기회를 위한 기술 섹터 리서치
# (기술 섹터 - 긍정적 전망, AI 채택/클라우드 성장/디지털 전환 주도, 비중 확대 5% 증가 추천)
response = await investment_agent.run(
    "Research market sector: 'Technology' with 'positive' outlook, key drivers 'AI adoption, cloud growth, digital transformation', "
    "recommendation 'overweight - increase allocation by 5%'.",
    memory=agentcore_memory
)

print("기술 섹터 리서치:")
print(response)

# 분산 투자를 위한 헬스케어 섹터 리서치
# (헬스케어 섹터 - 중립 전망, 고령화 인구/신약 혁신/규제 변화 주도, 현재 배분 유지 추천)
response = await investment_agent.run(
    "Research market sector: 'Healthcare' with 'neutral' outlook, key drivers 'aging demographics, drug innovation, regulatory changes', "
    "recommendation 'maintain - current allocation appropriate'.",
    memory=agentcore_memory
)

print("헬스케어 섹터 리서치:")
print(response)

### 테스트 6: 투자 추천

In [ ]:
# 구체적인 투자 추천 생성
# (QQQ(나스닥 ETF) 매수 추천 - 섹터 리서치에 따라 기술 노출 증가, 중간 위험 프로필에 부합, 목표 배분 8%)
response = await investment_agent.run(
    "Generate investment recommendation: security 'QQQ (Nasdaq ETF)', action 'BUY', "
    "rationale 'increase tech exposure per sector research, aligns with moderate risk profile', target allocation '8%'.",
    memory=agentcore_memory
)

print("QQQ 투자 추천:")
print(response)

# (VGIT(중기 국채 ETF) 축소 추천 - 기술 배분 자금 확보를 위한 리밸런싱, 듀레이션 위험 관리 유지, 목표 배분 12%)
response = await investment_agent.run(
    "Generate investment recommendation: security 'VGIT (Intermediate Treasury ETF)', action 'REDUCE', "
    "rationale 'rebalance to fund tech allocation, maintain duration risk management', target allocation '12%'.",
    memory=agentcore_memory
)

print("VGIT 리밸런싱 추천:")
print(response)

### 테스트 7: 위험 지표 회상 및 분석

In [ ]:
# 위험 지표 메모리 및 해석 테스트
# (Robert의 현재 위험 지표와 샤프 비율/베타가 중간 위험 허용도에 어떻게 부합하는지 질문)
response = await investment_agent.run(
    "What were Robert's current risk metrics? How does the Sharpe ratio and Beta align with his moderate risk tolerance?",
    memory=agentcore_memory
)

print("위험 지표 분석:")
print(response)
print("\n예상 결과: VaR -$125K, 샤프 1.15, 베타 0.85, 변동성 12.3% - 중간 위험에 적합")

### 테스트 8: 규정 준수 확인

In [ ]:
# 추천에 대한 규정 준수 확인
# (수탁자 의무 - 최선의 이익: 준수, 추천이 고객의 위험 프로필 및 투자 목표에 부합)
response = await investment_agent.run(
    "Check regulatory compliance: rule type 'Fiduciary Duty - Best Interest', compliance status 'COMPLIANT', "
    "notes 'recommendations align with client risk profile and investment objectives'.",
    memory=agentcore_memory
)

print("수탁자 의무 준수:")
print(response)

# (포트폴리오 집중 한도: 준수, 단일 포지션 15% 미초과, 섹터 배분 가이드라인 내)
response = await investment_agent.run(
    "Check regulatory compliance: rule type 'Portfolio Concentration Limits', compliance status 'COMPLIANT', "
    "notes 'no single position exceeds 15%, sector allocation within guidelines'.",
    memory=agentcore_memory
)

print("집중도 준수:")
print(response)

### 테스트 9: 투자 근거 통합

In [ ]:
# 통합 투자 추론 테스트
# (섹터 리서치와 Robert의 프로필을 기반으로, QQQ 배분 증가를 추천한 이유와 위험 허용도/투자 목표와의 부합 여부 질문)
response = await investment_agent.run(
    "Based on my sector research and Robert's profile, why did I recommend increasing QQQ allocation? "
    "How does this align with his risk tolerance and investment goals?",
    memory=agentcore_memory
)

print("투자 근거:")
print(response)
print("\n예상 결과: 기술 섹터 긍정적 전망 + 중간 위험 허용도 + 15년 기간 = QQQ 증가")

In [ ]:
# 종합 자문 세션 요약
# (고객 프로필, 현재 포트폴리오 지표, 섹터 리서치 결과, 투자 추천 및 준수 상태의 완전한 자문 요약 요청)
response = await investment_agent.run(
    "Provide a complete advisory summary: client profile, current portfolio metrics, sector research findings, "
    "investment recommendations, and compliance status. Include rationale for all recommendations.",
    memory=agentcore_memory
)

print("종합 자문 요약:")
print(response)
print("\n예상 결과: Robert의 프로필, $2.5M 포트폴리오, 기술/헬스케어 리서치, QQQ/VGIT 추천 포함 전체 세션 세부 정보")

## 6단계: 세션 경계 테스트

다른 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다:

In [ ]:
# 다른 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="financial-advisor",
    memory_id=memory_id,
    session_id="different-advisory-session",  # 다른 세션 ID
    namespace="/investment-advisory/"
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
# (오늘 어떤 고객을 상담 중인지, 포트폴리오 가치와 투자 추천 내역 질문)
response = await investment_agent.run(
    "What clients am I advising today? What portfolio values and investment recommendations have I made?",
    memory=new_session_memory
)

print("세션 경계 테스트 (다른 세션):")
print(response)
print("\n예상 결과: 이전 세션에서의 회상이 제한적이거나 없음 (단기 메모리 경계)")

In [ ]:
# 원래 세션으로 돌아가서 지속성 확인
# (원래 세션으로 돌아와서 Robert Johnson의 정확한 위험 지표와 QQQ 추천 질문)
response = await investment_agent.run(
    "Back in my original session - what were Robert Johnson's exact risk metrics and my QQQ recommendation?",
    memory=agentcore_memory  # 원래 세션 메모리
)

print("원래 세션 복귀:")
print(response)
print("\n예상 결과: 샤프 1.15, 베타 0.85, QQQ 매수 8% 배분 전체 회상")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀들을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 세션 초반 정보를 회상할 수 있는지 확인"""
        has_content = len(response) > 50
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내에서 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동합니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하며, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 중요한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - 에이전트가 논의된 내용을 회상할 수 있는가?
response1 = await investment_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리 - 에이전트가 컨텍스트를 유지하는가?
response2 = await investment_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 기능 - 이전 컨텍스트와 연결할 수 있는가?
response3 = await investment_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

### 테스트 10: 종합 자문 요약

## 요약

이 노트북에서 다음을 시연했습니다:

- **단기 메모리 통합**: 세션 범위 투자 자문을 위해 LlamaIndex와 AgentCore 메모리 사용

- **금융 전용 도구**: 고객 프로파일링, 포트폴리오 분석, 위험 지표 및 투자 추천

- **투자 추론**: 어드바이저가 고객 프로필, 시장 조사 및 추천 근거를 기억

- **위험 관리**: 종합적인 위험 지표 추적 및 규정 준수 확인

- **세션 경계**: 서로 다른 고객 자문 세션 간의 메모리 격리

- **규정 준수**: 수탁자 의무 및 투자 가이드라인 준수

투자 포트폴리오 어드바이저는 단기 메모리가 어떻게 단일 고객 세션 내에서 포괄적인 금융 자문을 가능하게 하면서 서로 다른 고객 상담 간에 명확한 경계를 유지하는지 보여줍니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다:

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")